<a href="https://colab.research.google.com/github/mixxr/gcolab-ml/blob/main/funnel/04_drug_likeness_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Episode 4 — Drug-Likeness & Filtering
**Drug Discovery in Code** · [Life Sciences AI Applications]
In Episode 3 we pulled ~1,000 real EGFR inhibitors from ChEMBL. But being *active* isn't enough —
a molecule also has to behave like a **drug**: absorbable, not greasy, and free of nasty
substructures that trip up assays. Today we run two classic filters — **Lipinski's Rule of Five**
and **PAINS structural alerts** — to trim the pile down to the compounds actually worth pursuing.
> Educational only — not medical advice.

## Setup — load the Episode 3 dataset

In [ ]:
import pandas as pd, numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski

df = pd.read_csv("../data/egfr_chembl.csv")
len(df)

## 1. Compute the physicochemical properties
The Rule of Five is about four numbers: **molecular weight**, **LogP** (greasiness),
**H-bond donors**, and **H-bond acceptors**. RDKit computes all of them from the SMILES.

In [ ]:
def properties(smiles):
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        return None
    return pd.Series({
        "MolWt": Descriptors.MolWt(m),
        "LogP":  Descriptors.MolLogP(m),
        "HBD":   Lipinski.NumHDonors(m),
        "HBA":   Lipinski.NumHAcceptors(m),
    })

df = df.join(df.smiles.apply(properties))
df = df.dropna(subset=["MolWt"]).reset_index(drop=True)
df[["molecule_chembl_id","MolWt","LogP","HBD","HBA"]].head()

## 2. Lipinski's Rule of Five
The rule: **MolWt ≤ 500**, **LogP ≤ 5**, **HBD ≤ 5**, **HBA ≤ 10**. In practice a molecule is
allowed **one** violation and still counts as drug-like — so we count violations and keep
everything with at most one.

In [ ]:
def ro5_violations(r):
    return (int(r.MolWt > 500) + int(r.LogP > 5)
            + int(r.HBD > 5) + int(r.HBA > 10))

df["violations"] = df.apply(ro5_violations, axis=1)
df["ro5_pass"]   = df.violations <= 1

print(df.ro5_pass.sum(), "of", len(df), "pass the Rule of Five")

## 3. Structural alerts — PAINS
**PAINS** (Pan-Assay INterference compoundS) are substructures that show up as false hits in
assay after assay. RDKit ships a `FilterCatalog` for them — no need to write SMARTS by hand.

In [ ]:
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
catalog = FilterCatalog(params)

def has_pains(smiles):
    m = Chem.MolFromSmiles(smiles)
    return bool(m and catalog.HasMatch(m))

df["pains"] = df.smiles.apply(has_pains)
print(df.pains.sum(), "compounds carry a PAINS alert")

## 4. The filtering funnel
Chain the two filters: keep only compounds that pass the Rule of Five **and** carry no PAINS alert.

In [ ]:
clean = df[df.ro5_pass & ~df.pains].copy()

print(f"start:        {len(df):>5}")
print(f"pass Ro5:     {df.ro5_pass.sum():>5}")
print(f"clean set:    {len(clean):>5}")

# save the drug-like set for the episodes ahead
clean.to_csv("../data/egfr_druglike.csv", index=False)

## 5. See the property space
Plot every molecule in molecular-weight vs LogP space, with the Rule-of-Five box drawn on top —
the failures spill outside it.

In [ ]:
import matplotlib.pyplot as plt
ok = df.ro5_pass
plt.scatter(df.MolWt[ok],  df.LogP[ok],  s=8, alpha=.5, label="pass")
plt.scatter(df.MolWt[~ok], df.LogP[~ok], s=8, alpha=.6, c="red", label="fail")
plt.axvline(500, ls="--", c="orange"); plt.axhline(5, ls="--", c="orange")
plt.xlabel("MolWt"); plt.ylabel("LogP"); plt.legend(); plt.show()

From ~1,000 active molecules down to ~840 that are both drug-like and alert-free — a cleaner set
to search and model. **Filters are cheap; assays are expensive.** Removing obvious liabilities
early is how real campaigns avoid wasting effort.

## Recap
- **Active ≠ drug-like** — a good molecule also has to be absorbable and clean.
- **Rule of Five** flags poor oral-absorption profiles from four simple descriptors.
- **PAINS** alerts catch notorious assay-interfering substructures.
- We chained them into a funnel and saved the drug-like set.

**Next — Episode 5:** **similarity search** — take a known inhibitor and find look-alikes in our
set with Tanimoto + fingerprints.